# Optimize All Dataset Configs

Run CLASP latent Bayesian optimization for every configured dataset and save the best preprocessing, estimator, and graph settings to `data/optimized_params/`. Notebook 02 reads these files with `load_or_default_params(...)`, so this notebook provides the shared optimization baseline for visualization and evaluation.

Run `00_download_datasets.ipynb` first so the configured `.h5ad` inputs exist locally. The loop can be resumed: datasets with existing optimized parameter files are skipped unless `overwrite_existing = True`.

In [1]:
from __future__ import annotations

import gc
import os
from pathlib import Path
import traceback

import pandas as pd

from clasp.notebook_utils import (
    PAPER_DATASET_DOWNLOADS,
    dataset_config,
    make_clasp_optimization_objective,
    make_compact_search_space,
    make_estimator,
    optimization_search_space,
    optimized_params_path,
    preprocess_params,
    resolve_project_root,
    run_latent_bayesopt,
    save_best_optimization_result,
)

In [2]:
project_root = resolve_project_root()
os.chdir(project_root)

random_state = 0
selected_datasets = list(PAPER_DATASET_DOWNLOADS)

skip_missing_inputs = True
overwrite_existing = False
run_gplvm_refinement = True

embedding_method = "umap"
embedding_epochs = 60
invalid_score = -1e9
summary_path = project_root / "data" / "optimized_params" / "all_datasets_optimization_summary.csv"

# Keep this cap modest for all-dataset sweeps. Increase it for final high-quality runs.
fixed_preprocess_params = {"max_cells": 1000}

pca_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 12,
    "embedding_model": "pca",
    "acquisition": "ei",
    "batch_size": 1,
}

gplvm_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 12,
    "embedding_model": "gplvm",
    "acquisition": "ei",
    "batch_size": 1,
}

selected_datasets

['scib_pancreas',
 'scib_lung_atlas',
 'scib_immune_human',
 'scib_immune_human_mouse',
 'cellrank_bone_marrow',
 'cellrank_lung',
 'cellrank_pancreas',
 'cellrank_reprogramming_schiebinger',
 'cellrank_reprogramming_morris',
 'zebrafish']

In [3]:
def base_optimization_config(dataset: dict) -> tuple[dict, dict, dict, dict, dict, dict, dict]:
    base_preprocess_params = preprocess_params(dataset, n_top_genes=1500)
    base_estimator_params = {
        "n_components": 60,
    }
    base_graph_params = {
        **dataset.get("graph", {}),
        "n_neighbors": 15,
        "intra_fraction": 0.5,
        "n_inter_edges": 2,
        "metric": "euclidean",
        "assignment_quantile": 0.8,
        "hubness_correction": "csls",
        "hubness_k": 10,
        "rank_correction": True,
        "edge_weighting": "binary",
        "mutual_neighbors": False,
        "neighbor_mode": "distance",
        "symmetrize": True,
    }

    preprocess_search_space = {
        "n_top_genes": {"type": "int", "bounds": [500, 2000]},
    }
    estimator_search_space = {
        "n_components": {"type": "int", "bounds": [20, 100]},
    }
    graph_search_space = {
        "n_neighbors": {"type": "int", "bounds": [5, 35]},
        "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
        "n_inter_edges": {"type": "int", "bounds": [1, 6]},
        "assignment_quantile": {"type": "float", "bounds": [0.1, 1.0]},
        "hubness_k": {"type": "int", "bounds": [3, 20]},
        "rank_correction": {"type": "categorical", "values": [False, True]},
        "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
        "inter_edge_mode": {"type": "categorical", "values": ["propagate_neighbors", "assignment"]},
        "mutual_neighbors": {"type": "categorical", "values": [False, True]},
    }
    search_space = optimization_search_space(
        preprocess_search_space=preprocess_search_space,
        estimator_search_space=estimator_search_space,
        graph_search_space=graph_search_space,
    )
    return (
        base_preprocess_params,
        base_estimator_params,
        base_graph_params,
        preprocess_search_space,
        estimator_search_space,
        graph_search_space,
        search_space,
    )


compact_radii = {
    "n_top_genes": 300,
    "n_components": 20,
    "n_neighbors": 6,
    "intra_fraction": 0.1,
    "n_inter_edges": 2,
    "assignment_quantile": 0.15,
    "hubness_k": 4,
}

In [4]:
def optimize_dataset(dataset_name: str) -> dict:
    dataset = dataset_config(dataset_name, project_root=project_root)
    params_path = optimized_params_path(dataset_name, project_root=project_root)

    row = {
        "dataset": dataset_name,
        "input_path": str(dataset["input_path"]),
        "params_path": str(params_path),
        "status": "pending",
    }

    if params_path.exists() and not overwrite_existing:
        row["status"] = "skipped_existing"
        return row

    if not dataset["input_path"].exists():
        row["status"] = "missing_input"
        if skip_missing_inputs:
            return row
        raise FileNotFoundError(dataset["input_path"])

    (
        base_preprocess_params,
        base_estimator_params,
        base_graph_params,
        preprocess_search_space,
        estimator_search_space,
        graph_search_space,
        search_space,
    ) = base_optimization_config(dataset)

    raw_estimator = make_estimator(dataset, n_components=100, random_state=random_state)
    raw_adata = raw_estimator.to_data(dataset["input_path"])

    objective = make_clasp_optimization_objective(
        dataset=dataset,
        raw_adata=raw_adata,
        base_preprocess_params=base_preprocess_params,
        fixed_preprocess_params=fixed_preprocess_params,
        base_estimator_params=base_estimator_params,
        base_graph_params=base_graph_params,
        preprocess_search_space=preprocess_search_space,
        estimator_search_space=estimator_search_space,
        graph_search_space=graph_search_space,
        random_state=random_state,
        embedding_method=embedding_method,
        embedding_epochs=embedding_epochs,
        invalid_score=invalid_score,
    )

    pca_result = run_latent_bayesopt(
        objective,
        search_space,
        **pca_bo_settings,
        random_state=random_state,
        invalid_score=invalid_score,
    )

    optimization_results = {"pca": pca_result}
    gplvm_result = None
    if run_gplvm_refinement:
        compact_search_space = make_compact_search_space(
            search_space,
            pca_result["best_params"],
            compact_radii,
            fix_categoricals=True,
        )
        gplvm_result = run_latent_bayesopt(
            objective,
            compact_search_space,
            **gplvm_bo_settings,
            random_state=random_state + 1,
            invalid_score=invalid_score,
        )
        optimization_results["gplvm"] = gplvm_result

    params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params = save_best_optimization_result(
        dataset_name=dataset_name,
        optimization_results=optimization_results,
        base_preprocess_params=base_preprocess_params,
        fixed_preprocess_params=fixed_preprocess_params,
        base_estimator_params=base_estimator_params,
        base_graph_params=base_graph_params,
        preprocess_search_space=preprocess_search_space,
        estimator_search_space=estimator_search_space,
        graph_search_space=graph_search_space,
        random_state=random_state,
        project_root=project_root,
    )

    best_model = "gplvm" if gplvm_result is not None and gplvm_result["best_score"] >= pca_result["best_score"] else "pca"
    row.update(
        {
            "status": "optimized",
            "best_model": best_model,
            "best_score": max(result["best_score"] for result in optimization_results.values()),
            "pca_best_score": pca_result["best_score"],
            "gplvm_best_score": None if gplvm_result is None else gplvm_result["best_score"],
            "optimized_preprocess_params": optimized_preprocess_params,
            "optimized_estimator_params": optimized_estimator_params,
            "optimized_graph_params": optimized_graph_params,
        }
    )

    del raw_adata, raw_estimator, objective, pca_result, gplvm_result, optimization_results
    gc.collect()
    return row

In [ ]:
summary_rows = []

for index, dataset_name in enumerate(selected_datasets, start=1):
    print(f"[{index}/{len(selected_datasets)}] {dataset_name}")
    try:
        row = optimize_dataset(dataset_name)
    except Exception as exc:
        row = {
            "dataset": dataset_name,
            "status": "failed",
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        }
        print(row["traceback"])
    summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(summary_path, index=False)
    display(summary.tail(1))

summary

[1/10] scib_pancreas


,dataset,input_path,params_path,status,best_model,best_score,pca_best_score,gplvm_best_score,optimized_preprocess_params,optimized_estimator_params,optimized_graph_params
0,scib_pancreas,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,optimized,gplvm,1.07715,1.05435,1.07715,"{'n_top_genes': 830, 'max_cells': 1000, 'min_c...",{'n_components': 66},"{'n_neighbors': 21, 'intra_fraction': 0.536881..."


[2/10] scib_lung_atlas


The JSON parameter files written by this run are consumed directly by notebook 02:

```python
optimized_params = load_or_default_params(selected_dataset, dataset)
```

The CSV summary is only a run log; the JSON files in `data/optimized_params/` are the source of truth for downstream visualization and evaluation.